# AI detection tools evaluation
## Dataset download

In [1]:
from datasets import load_dataset
ds = load_dataset("silentone0725/ai-human-text-detection-v1")

Repo card metadata block was not found. Setting CardData to empty.


In [2]:
ds.save_to_disk("ai_human_text_detection_data")

Saving the dataset (0/1 shards):   0%|          | 0/36744 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/7874 [00:00<?, ? examples/s]

Saving the dataset (0/1 shards):   0%|          | 0/7874 [00:00<?, ? examples/s]

In [3]:
data = ds["train"]

TEXT_COL = "text"
LABEL_COL = "label"

ai_texts = []
human_texts = []

for item in data:

    text = item[TEXT_COL].strip()
    if not text:
        continue
    text = text.strip()
    label = item[LABEL_COL]

    if len(text.split()) < 50:
        continue

    if label == "ai" and len(ai_texts) < 25:
        ai_texts.append(text)
    elif label == "human" and len(human_texts) < 25:
        human_texts.append(text)

    if len(ai_texts) == 25 and len(human_texts) == 25:
        break

def save_to_file(texts, filename):
    with open(filename, "w", encoding="utf-8") as f:
        for i, text in enumerate(texts, start=1):
            f.write(f"{i}\n{text}\n\n")

save_to_file(ai_texts, "tools_test_data/ai_texts.txt")
save_to_file(human_texts, "tools_test_data/human_texts.txt")

## Metrics
### GPTZero

In [22]:
import re
import pandas as pd

def parse_file(path):
    with open(path, "r", encoding="utf-8") as f:
        content = f.read()

    entries = re.split(r'\n(?=\d+[.,])', content)

    rows = []

    for entry in entries[1:]:
        entry = entry.strip()
        if not entry:
            continue

        try:
            first_comma = entry.find(',')
            second_comma = entry.find(',', first_comma + 1)

            id_ = entry[:first_comma].strip()
            text_and_rest = entry[first_comma+1:]

            last_comma = text_and_rest.rfind(',')
            second_last_comma = text_and_rest.rfind(',', 0, last_comma)

            text = text_and_rest[:second_last_comma].strip().strip('"')
            label = text_and_rest[second_last_comma+1:last_comma].strip()
            score = text_and_rest[last_comma+1:].strip()

            rows.append({
                "id": int(id_),
                "text": text,
                "true_label": label,
                "score": float(score)
            })

        except Exception as e:
            print(f"Skipping malformed entry:\n{entry[:100]}...\nError: {e}\n")

    return pd.DataFrame(rows)


df = parse_file("results/gprzero.csv")

print(df.head())
print(f"Loaded rows: {len(df)}")

   id                                               text true_label  score
0   1  We don't actually know what color dinosaurs li...         ai   0.00
1   2  A ship of the line was a type of warship that ...         ai   1.00
2   3  There are many complex factors that have contr...         ai   0.38
3   4  Elijah McCoy (1844-1929) was an African Americ...         ai   0.81
4   5  It's true that China is a major player in the ...         ai   0.05
Loaded rows: 30


In [23]:
from sklearn.metrics import classification_report, confusion_matrix

df["pred_label"] = df["score"].apply(lambda x: "ai" if x >= 0.5 else "human")

y_true = df["true_label"]
y_pred = df["pred_label"]

print(classification_report(y_true, y_pred))

cm = confusion_matrix(y_true, y_pred, labels=["human", "ai"])

tn, fp, fn, tp = cm.ravel()

print("\n=== CONFUSION MATRIX ===")
print(cm)

fpr = fp / (fp + tn) if (fp + tn) > 0 else 0

print("\n=== ADDITIONAL ===")
print(f"False Positive Rate (FPR): {fpr:.3f}")

              precision    recall  f1-score   support

          ai       1.00      0.67      0.80        15
       human       0.75      1.00      0.86        15

    accuracy                           0.83        30
   macro avg       0.88      0.83      0.83        30
weighted avg       0.88      0.83      0.83        30


=== CONFUSION MATRIX ===
[[15  0]
 [ 5 10]]

=== ADDITIONAL ===
False Positive Rate (FPR): 0.000
